In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

# ── Connect ────────────────────────────────────────────────────────────────
session = get_active_session()

# ── Load FEAT_FINAL ────────────────────────────────────────────────────────
df = session.table('FRAUD_DB.FEATURES.FEAT_FINAL').to_pandas()
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["ISFRAUD"].mean() * 100:.2f}%')

# ── Sort by time ───────────────────────────────────────────────────────────
# CRITICAL: never random-split a time-series fraud dataset.
# Random splitting leaks future fraud patterns into training — model appears
# better than it actually is in production.
# Time-based split simulates real deployment: always predicting on unseen future.
df = df.sort_values('TXN_DAY_NUMBER').reset_index(drop=True)

# ── Sentinel imputation ────────────────────────────────────────────────────
# Handles V1-V339 nulls (not coalesced in SQL) and any other remaining nulls.
# -999 chosen because:
#   1. 0 is a valid value in many columns — using it as sentinel creates false signal
#   2. Mean imputation hides the missingness pattern itself
#   3. Tree models create a separate branch for -999 so the model learns
#      "when this field is missing, behaviour is X" — missingness becomes signal
df.fillna(-999, inplace=True)

# Fix string columns: fillna converts NaN to float -999 in object columns.
# Restore 'unknown' so label encoding handles them consistently.
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('-999', 'unknown').replace(-999, 'unknown')

# ── Feature lists ──────────────────────────────────────────────────────────
CATEGORICAL = [
    'PRODUCTCD',
    'CARD4_CLEAN',
    'CARD6_CLEAN',
    'P_EMAIL_GROUP',
    'R_EMAIL_GROUP',
    'DEVICE_TYPE',
    'OS_GROUP',
    'OS_VERSION_GROUP',
    'BROWSER_FAMILY',
    'CHROME_VERSION_BUCKET',
    'DEVICE_MANUFACTURER',
    'SAMSUNG_TIER',
    'M4_CLEAN',
    'ID_12',
    'ID_15',
    'ID_16',
    'ID_23',
    'ID_27',
    'ID_28',
    'ID_29',
]

NUMERIC = [
    # Time features
    'TXN_HOUR', 'TXN_DAY_OF_WEEK',
    'IS_NIGHT_TRANSACTION',
    # Amount features
    'LOG_TXN_AMT', 'AMT_DECIMAL_PART',
    # Card and address
    'CARD1', 'CARD2', 'CARD3', 'CARD5',
    'ADDR1_CLEAN', 'ADDR2_CLEAN', 'DIST1_CLEAN', 'DIST2_CLEAN',
    # Match flags M1-M3, M5-M9 (binary). M4 is categorical above.
    'M1_FLAG', 'M2_FLAG', 'M3_FLAG',
    'M5_FLAG', 'M6_FLAG', 'M7_FLAG', 'M8_FLAG', 'M9_FLAG',
    # Count columns C1-C14 (how many addresses/devices/etc per card)
    'C1',  'C2',  'C3',  'C4',  'C5',  'C6',  'C7',
    'C8',  'C9',  'C10', 'C11', 'C12', 'C13', 'C14',
    # Time-gap columns D1-D15 (days between events)
    'D1',
    'D2',  'D3',  'D4',  'D5',  'D6',  'D7',  'D8',
    'D9',  'D10', 'D11', 'D12', 'D13', 'D14', 'D15',
    # Device and identity flags
    'HAS_IDENTITY',
    'IS_EXOTIC_BROWSER',
    'NO_BROWSER_OS_SIGNAL',
    'IS_OLD_ANDROID_BUDGET_DEVICE',
    # Identity numeric id_01-id_11
    'ID_01', 'ID_02', 'ID_03', 'ID_04', 'ID_05',
    'ID_06', 'ID_07', 'ID_08', 'ID_09', 'ID_10', 'ID_11',
    # Identity numeric id_13, id_14
    'ID_13', 'ID_14',
    # Identity numeric id_17-id_22, id_24-id_26
    'ID_17', 'ID_18', 'ID_19', 'ID_20', 'ID_21', 'ID_22',
    'ID_24', 'ID_25', 'ID_26',
    # Identity numeric id_32
    'ID_32',
    # Screen resolution split from id_33
    'SCREEN_WIDTH', 'SCREEN_HEIGHT',
    # Match status extracted from id_34 (ordinal: -1, 0, 1, 2)
    'ID_34_MATCH_STATUS',
    # Identity boolean flags id_35-id_38
    'ID_35_FLAG', 'ID_36_FLAG', 'ID_37_FLAG', 'ID_38_FLAG',
    # Velocity features (rolling window aggregates)
    'CARD1_TXN_COUNT_7D', 'CARD1_AMT_SUM_7D', 'CARD1_AVG_AMT_30D',
    'AMT_TO_CARD_AVG_RATIO', 'EMAIL_TXN_COUNT_7D', 'UID_TXN_COUNT_30D',
]

# V1-V339: added dynamically so list stays maintainable.
V_COLS = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
NUMERIC += V_COLS

# ── Label encode categoricals ──────────────────────────────────────────────
le = LabelEncoder()
for col in CATEGORICAL:
    df[col] = le.fit_transform(df[col].astype(str))

FEATURES = NUMERIC + CATEGORICAL
TARGET   = 'ISFRAUD'

X = df[FEATURES]
y = df[TARGET]

# ── Time-based train/validation split ─────────────────────────────────────
split_idx      = int(len(df) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Train fraud rate: {y_train.mean() * 100:.2f}%')
print(f'Val fraud rate:   {y_val.mean() * 100:.2f}%')

# ── Class imbalance handling ───────────────────────────────────────────────
fraud_count      = y_train.sum()
legit_count      = len(y_train) - fraud_count
scale_pos_weight = legit_count / fraud_count
print(f'scale_pos_weight: {scale_pos_weight:.2f}')


# =============================================================================
# MODEL 1 — XGBOOST
# =============================================================================
print('\n--- Training XGBoost ---')
xgb_model = XGBClassifier(
    n_estimators          = 1000,
    max_depth             = 8,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.4,
    scale_pos_weight      = scale_pos_weight,
    eval_metric           = 'auc',
    early_stopping_rounds = 50,
    random_state          = 42,
    n_jobs                = -1,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)
xgb_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_roc   = roc_auc_score(y_val, xgb_proba)
xgb_pr    = average_precision_score(y_val, xgb_proba)
print(f'XGBoost  ROC-AUC: {xgb_roc:.4f}  PR-AUC: {xgb_pr:.4f}')


# =============================================================================
# MODEL 2 — LIGHTGBM
# =============================================================================
print('\n--- Training LightGBM ---')
lgb_model = LGBMClassifier(
    n_estimators      = 1000,
    max_depth         = 8,
    learning_rate     = 0.05,
    num_leaves        = 63,
    subsample         = 0.8,
    colsample_bytree  = 0.4,
    min_child_samples = 20,
    class_weight      = 'balanced',
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ],
)
lgb_proba = lgb_model.predict_proba(X_val)[:, 1]
lgb_roc   = roc_auc_score(y_val, lgb_proba)
lgb_pr    = average_precision_score(y_val, lgb_proba)
print(f'LightGBM ROC-AUC: {lgb_roc:.4f}  PR-AUC: {lgb_pr:.4f}')


# =============================================================================
# MODEL COMPARISON
# =============================================================================
print('\n--- Model Comparison ---')
results = pd.DataFrame({
    'Model':   ['XGBoost', 'LightGBM'],
    'ROC_AUC': [xgb_roc,  lgb_roc],
    'PR_AUC':  [xgb_pr,   lgb_pr],
})
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2196F3', '#4CAF50']

axes[0].bar(results['Model'], results['ROC_AUC'], color=colors)
axes[0].set_title('ROC-AUC Comparison\n(higher is better, baseline = 0.5)')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_ylim(0.85, 0.95)
axes[0].axhline(y=0.88, color='red', linestyle='--', label='Target (0.88)')
axes[0].legend()
for i, v in enumerate(results['ROC_AUC']):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=11)

axes[1].bar(results['Model'], results['PR_AUC'], color=colors)
axes[1].set_title('PR-AUC Comparison\n(higher is better, baseline = 0.035)')
axes[1].set_ylabel('PR-AUC')
axes[1].set_ylim(0, 0.8)
axes[1].axhline(y=0.035, color='red', linestyle='--',
                label='Baseline (random at 3.5% fraud rate)')
axes[1].legend()
for i, v in enumerate(results['PR_AUC']):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=11)

plt.suptitle('Model Comparison — IEEE-CIS Fraud Detection', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()


# =============================================================================
# PICK BEST MODEL
# =============================================================================
best_name  = results.loc[results['ROC_AUC'].idxmax(), 'Model']
best_proba = {'XGBoost': xgb_proba, 'LightGBM': lgb_proba}[best_name]
best_model = {'XGBoost': xgb_model, 'LightGBM': lgb_model}[best_name]
print(f'\nBest model by ROC-AUC: {best_name}')

In [ ]:
# =============================================================================
# PRECISION-RECALL CURVES — BOTH MODELS
# =============================================================================
fig, ax = plt.subplots(figsize=(9, 6))

for proba, label, color in [
    (xgb_proba, f'XGBoost  (PR-AUC={xgb_pr:.3f})', '#2196F3'),
    (lgb_proba, f'LightGBM (PR-AUC={lgb_pr:.3f})', '#4CAF50'),
]:
    p, r, _ = precision_recall_curve(y_val, proba)
    ax.plot(r, p, label=label, color=color)

ax.axhline(y=0.035, color='gray', linestyle='--',
           label='Baseline (random classifier at 3.5% fraud rate)')
ax.set_xlabel('Recall — proportion of actual fraud caught')
ax.set_ylabel('Precision — of flagged transactions, how many are real fraud')
ax.set_title('Precision-Recall Curves — XGBoost vs LightGBM')
ax.legend()
plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# FRAUD SCORE DISTRIBUTION — BEST MODEL
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(best_proba[y_val == 0], bins=50, alpha=0.7,
        label='Legitimate', color='steelblue', density=True)
ax.hist(best_proba[y_val == 1], bins=50, alpha=0.7,
        label='Fraud', color='crimson', density=True)
ax.set_xlabel('Fraud Score (model output probability)')
ax.set_ylabel('Density')
ax.set_title(f'Fraud Score Distribution — {best_name}\n'
             f'Clear separation between peaks = model is confident')
ax.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# CONFUSION MATRIX AT 0.5 THRESHOLD
# =============================================================================
y_pred = (best_proba >= 0.5).astype(int)
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Legitimate', 'Fraud']
)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix at 0.5 Threshold — {best_name}\n'
             f'Bottom-left = missed fraud (costly). '
             f'Top-right = false alarms (customer friction).')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nAt 0.5 threshold:')
print(f'  True Positives  (fraud caught):        {tp:,}')
print(f'  False Negatives (fraud missed):        {fn:,}')
print(f'  False Positives (legitimate flagged):  {fp:,}')
print(f'  True Negatives  (legitimate passed):   {tn:,}')
print(f'  Precision: {tp / (tp + fp):.3f}')
print(f'  Recall:    {tp / (tp + fn):.3f}')

In [ ]:
# =============================================================================
# THRESHOLD ANALYSIS
# =============================================================================
precision_arr, recall_arr, thresholds_arr = precision_recall_curve(
    y_val, best_proba
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds_arr, precision_arr[:-1], label='Precision', color='#2196F3')
ax.plot(thresholds_arr, recall_arr[:-1],   label='Recall',    color='crimson')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Precision vs Recall at Different Thresholds — {best_name}\n'
             f'Move threshold left = catch more fraud. '
             f'Move right = reduce false alarms.')
ax.axvline(x=0.5, color='gray', linestyle='--', label='Default threshold (0.5)')
ax.legend()
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# FEATURE IMPORTANCE — TOP 20
# =============================================================================
importances = pd.DataFrame({
    'feature':    FEATURES,
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importances['feature'][::-1],
        importances['importance'][::-1],
        color='#2196F3')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title(f'Top 20 Features — {best_name}\n'
             f'C columns = counting features. '
             f'D columns = time-gap features.')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# SHAP VALUES — BEST MODEL
# =============================================================================
import shap
explainer   = shap.TreeExplainer(best_model)
shap_sample = X_val.sample(5000, random_state=42)
shap_values = explainer.shap_values(shap_sample)

plt.figure(figsize=(10, 10))
shap.summary_plot(
    shap_values,
    shap_sample,
    feature_names=FEATURES,
    max_display=20,
    show=False,
)
plt.title(f'SHAP Feature Impact — {best_name}\n'
          f'Red = high feature value, Blue = low. '
          f'Right of centre = pushes towards fraud prediction.')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150)
plt.show()

In [ ]:
# THRESHOLD ANALYSIS TABLE — write to Snowflake for Power BI Strategy page
# =============================================================================
# Pre-calculates precision, recall, and cost at each threshold 0.1 to 0.9.
# Power BI reads this table and uses a slicer on THRESHOLD to show
# how model performance changes at each decision point.
# Business cost assumptions (adjust to realistic figures for your context)
AVG_FRAUD_LOSS = 250 # average financial loss per missed fraud transaction ($)
INVESTIGATION_COST = 15 # analyst cost per false alarm investigated ($)
thresholds = [round(t, 1) for t in np.arange(0.1, 1.0, 0.1)]
rows = []
for t in thresholds:
    pred = (best_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    precision_t = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0
    recall_t = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0
    rows.append({
        'THRESHOLD': t,
        'TRUE_POSITIVES': int(tp),
        'FALSE_POSITIVES': int(fp),
        'FALSE_NEGATIVES': int(fn),
        'TRUE_NEGATIVES': int(tn),
        'PRECISION': precision_t,
        'RECALL': recall_t,
        'FRAUD_CAUGHT_PCT': round(recall_t * 100, 2),
        'FALSE_ALARM_RATE_PCT': round(fp / (fp + tn) * 100, 4),
        'COST_MISSED_FRAUD': int(fn) * AVG_FRAUD_LOSS,
        'COST_FALSE_ALARMS': int(fp) * INVESTIGATION_COST,
        'TOTAL_COST': int(fn) * AVG_FRAUD_LOSS + int(fp) * INVESTIGATION_COST,
    })
threshold_df = pd.DataFrame(rows)
print(threshold_df.to_string(index=False))
session.sql('USE DATABASE FRAUD_DB').collect()
session.sql('USE SCHEMA ML').collect()
threshold_sp = session.create_dataframe(threshold_df)
threshold_sp.write.mode('overwrite').save_as_table('FRAUD_DB.ML.THRESHOLD_ANALYSIS')
print('Threshold analysis written to FRAUD_DB.ML.THRESHOLD_ANALYSIS')

In [ ]:
# =============================================================================
# WRITE PREDICTIONS TO SNOWFLAKE
# =============================================================================
session.sql('USE DATABASE FRAUD_DB').collect()
session.sql('USE SCHEMA ML').collect()
session.sql('USE WAREHOUSE FRAUD_WH').collect()

val_df = df.iloc[split_idx:].copy()
val_df['FRAUD_SCORE']     = best_proba
val_df['FRAUD_PREDICTED'] = (best_proba >= 0.5).astype(int)
val_df['MODEL_USED']      = best_name
val_df['RISK_TIER'] = pd.cut(
    val_df['FRAUD_SCORE'],
    bins   = [0, 0.3, 0.6, 0.8, 1.0],
    labels = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'],
)

predictions_sp = session.create_dataframe(
    val_df[[
        'TRANSACTIONID', 'FRAUD_SCORE', 'FRAUD_PREDICTED',
        'RISK_TIER', 'ISFRAUD', 'MODEL_USED',
        'IS_EXOTIC_BROWSER', 'NO_BROWSER_OS_SIGNAL',
        'DEVICE_TYPE', 'BROWSER_FAMILY', 'OS_VERSION_GROUP',
        'DEVICE_MANUFACTURER', 'P_EMAIL_GROUP',
        'TXN_HOUR', 'TXN_TIME_BUCKET',
        'CARD1_TXN_COUNT_7D', 'AMT_TO_CARD_AVG_RATIO',
    ]]
)
predictions_sp.write.mode('overwrite').save_as_table('FRAUD_DB.ML.PREDICTIONS')
print(f'\nPredictions written to FRAUD_DB.ML.PREDICTIONS')
print(f'Model used: {best_name}')
print(f'Validation rows: {len(val_df):,}')

if best_name == 'XGBoost':
    best_model.save_model('xgb_model.json')
else:
    best_model.booster_.save_model('lgb_model.txt')

print('Done.')

In [ ]:
# DOWNLOAD SHAP MODEL

# import pickle
# import os

# # 1. Define the model file name
# model_filename = 'your_fraud_model.pkl'
# local_temp_path = f'/tmp/{model_filename}'

# # 2. Serialize the model into a pickle file in Snowflake's temporary environment
# with open(local_temp_path, 'wb') as f:
#     pickle.dump(best_model, f)

# # 3. Create an internal stage if you don't have one already
# session.sql("CREATE STAGE IF NOT EXISTS ML_MODEL_STAGE").collect()

# # 4. Upload the pickle file from /tmp to your Snowflake stage
# session.file.put(
#     local_temp_path, 
#     '@ML_MODEL_STAGE', 
#     auto_compress=False, 
#     overwrite=True
# )
# print("Model successfully uploaded to @ML_MODEL_STAGE!")